<div align="center">

# Trivia Night with KONASH

**Train a 7B model to answer multi-constraint trivia by searching Wikipedia**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/konaequity/openkona/blob/main/notebooks/trivia_night.ipynb)

</div>

---

This notebook trains a **Qwen 3.5 7B** model to answer multi-constraint trivia questions using KONASH's iterative off-policy RL pipeline. The agent learns to search a Wikipedia corpus, retrieve relevant evidence, and synthesize accurate answers.

**What you'll build:**
- A knowledge agent that searches Wikipedia to answer trivia questions
- Trained with OAPL (Offline Advantage-weighted Policy Learning) on a single GPU
- 2 training iterations with automatic curriculum synthesis

**Requirements:**
- A Colab GPU runtime — **not enabled by default!** Go to **Runtime > Change runtime type > T4 GPU** before running.
- ~30 minutes for full training

---

## 0. Setup & Installation

In [ ]:
!pip install -q konash[train] wikipedia-api
print("konash installed successfully.")

In [ ]:
import os
import json
import time
import random
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch

# Check GPU availability
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime > Change runtime type > Select GPU.")

## 1. Build the Wikipedia Trivia Corpus

We fetch a curated set of Wikipedia articles spanning history, science, geography, literature, and pop culture — the pillars of any good trivia night. Each article becomes a searchable chunk in our FAISS index.

In [ ]:
import urllib.request
import urllib.parse


def fetch_wikipedia_article(title: str) -> str:
    """Fetch plain-text extract of a Wikipedia article via the API."""
    params = urllib.parse.urlencode({
        "action": "query",
        "titles": title,
        "prop": "extracts",
        "explaintext": "1",
        "exsectionformat": "plain",
        "format": "json",
    })
    url = f"https://en.wikipedia.org/w/api.php?{params}"
    req = urllib.request.Request(url, headers={"User-Agent": "KONASH-TriviaNotebook/1.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())
    pages = data.get("query", {}).get("pages", {})
    for page in pages.values():
        return page.get("extract", "")
    return ""


# Diverse trivia-worthy topics across categories
TRIVIA_TOPICS = [
    # History
    "Ancient Rome", "French Revolution", "Silk Road",
    "Space Race", "Rosetta Stone",
    # Science
    "Periodic table", "Theory of relativity", "DNA",
    "Black hole", "Photosynthesis",
    # Geography
    "Amazon River", "Mount Everest", "Great Barrier Reef",
    "Sahara", "Ring of Fire",
    # Literature & Arts
    "William Shakespeare", "Leonardo da Vinci",
    "The Great Gatsby", "Odyssey",
    # Pop Culture & Sports
    "Olympic Games", "FIFA World Cup", "Academy Awards",
    "Beatles", "Chess",
    # Miscellaneous
    "Human trafficking",  # replaced with less niche topic below
]
# Fix the last topic to something more trivia-appropriate
TRIVIA_TOPICS[-1] = "Seven Wonders of the Ancient World"

print(f"Fetching {len(TRIVIA_TOPICS)} Wikipedia articles...")
corpus_documents = []
for i, topic in enumerate(TRIVIA_TOPICS):
    text = fetch_wikipedia_article(topic)
    if text and len(text) > 200:
        # Truncate very long articles to first ~3000 words for manageable corpus size
        words = text.split()
        if len(words) > 3000:
            text = " ".join(words[:3000])
        corpus_documents.append({"text": text, "doc_id": topic})
        print(f"  [{i+1}/{len(TRIVIA_TOPICS)}] {topic}: {len(text):,} chars")
    else:
        print(f"  [{i+1}/{len(TRIVIA_TOPICS)}] {topic}: SKIPPED (too short)")

print(f"\nCorpus: {len(corpus_documents)} articles, "
      f"{sum(len(d['text']) for d in corpus_documents):,} total chars")

## 2. Write Corpus to Disk & Initialize KONASH

KONASH's `Corpus` class handles chunking, embedding, and FAISS indexing. We write the articles as plain text files and let the framework do the rest.

In [ ]:
# Write corpus to disk so KONASH Corpus can ingest it
CORPUS_DIR = "./trivia_corpus"
os.makedirs(CORPUS_DIR, exist_ok=True)

for doc in corpus_documents:
    # Sanitize filename
    fname = doc["doc_id"].replace(" ", "_").replace("/", "_") + ".txt"
    with open(os.path.join(CORPUS_DIR, fname), "w") as f:
        f.write(doc["text"])

print(f"Wrote {len(corpus_documents)} files to {CORPUS_DIR}/")

In [ ]:
from konash.corpus import Corpus

corpus = Corpus(
    CORPUS_DIR,
    chunk_size=384,     # Slightly smaller chunks for trivia (precise retrieval)
    chunk_overlap=48,
)
corpus.ingest()

print(f"Indexed {corpus.num_documents} chunks")
print(f"\nSanity check — searching for 'Shakespeare plays':")
results = corpus.search("Shakespeare plays", top_k=3)
for r in results:
    print(f"  score={r.get('score', 0):.3f}  {r['text'][:100]}...")

## 3. Seed Trivia Questions

We provide a small set of hand-crafted **multi-constraint** trivia questions as seeds. These demonstrate the style of questions we want the model to learn: questions that require synthesizing facts from one or more passages.

In [ ]:
from konash.synthesis.qa import SyntheticExample

SEED_TRIVIA = [
    SyntheticExample(
        question="What ancient trade route connected the Roman Empire to the Han Dynasty, and what was its primary commodity?",
        answer="The Silk Road connected the Roman Empire to the Han Dynasty of China. Its primary commodity was silk, though spices, metals, and ideas also traveled along its routes.",
        citations=["Silk Road", "Ancient Rome"],
    ),
    SyntheticExample(
        question="Which mountain is the tallest above sea level, and in which mountain range is it located?",
        answer="Mount Everest, standing at 8,849 meters (29,032 ft), is the tallest mountain above sea level. It is located in the Himalayas on the border between Nepal and Tibet.",
        citations=["Mount Everest"],
    ),
    SyntheticExample(
        question="Which artist painted the Mona Lisa and also designed flying machines centuries before the Wright brothers?",
        answer="Leonardo da Vinci painted the Mona Lisa (c. 1503-1519) and designed numerous flying machines, including ornithopters and aerial screws, as documented in his notebooks.",
        citations=["Leonardo da Vinci"],
    ),
    SyntheticExample(
        question="What molecule carries genetic information in all living organisms, and who discovered its double-helix structure?",
        answer="DNA (deoxyribonucleic acid) carries genetic information. Its double-helix structure was discovered by James Watson and Francis Crick in 1953, building on X-ray diffraction work by Rosalind Franklin.",
        citations=["DNA"],
    ),
    SyntheticExample(
        question="Which board game has had the most world champions from Russia, and what was the longest recorded championship match in its history?",
        answer="Chess. Russia (and the former Soviet Union) has produced the most world chess champions. The longest championship match was the 1984 Karpov-Kasparov match, which lasted 48 games over 5 months before being controversially terminated.",
        citations=["Chess"],
    ),
    SyntheticExample(
        question="What natural process converts sunlight into chemical energy in plants, and what gas does it release as a byproduct?",
        answer="Photosynthesis converts sunlight into chemical energy (glucose) in plants. It releases oxygen (O2) as a byproduct, produced from the splitting of water molecules.",
        citations=["Photosynthesis"],
    ),
]

print(f"{len(SEED_TRIVIA)} seed trivia questions:")
for i, ex in enumerate(SEED_TRIVIA, 1):
    print(f"\n  [{i}] Q: {ex.question}")
    print(f"      A: {ex.answer[:100]}...")

## 4. Held-Out Evaluation Set

These questions are reserved for testing. The deduplication step ensures no training question is too similar to these, preventing contamination.

In [ ]:
EVAL_TRIVIA = [
    {
        "question": "What event, starting in 1789, led to the execution of King Louis XVI and the rise of Napoleon Bonaparte?",
        "answer": "The French Revolution",
    },
    {
        "question": "What is the world's largest coral reef system, and in which country's waters is it primarily located?",
        "answer": "The Great Barrier Reef, located off the coast of Queensland, Australia.",
    },
    {
        "question": "Which physicist formulated the theory of general relativity, and what phenomenon does it describe?",
        "answer": "Albert Einstein formulated general relativity, which describes gravity as the curvature of spacetime caused by mass and energy.",
    },
    {
        "question": "What is the longest river in South America, and roughly how long is it?",
        "answer": "The Amazon River, approximately 6,400 km (4,000 miles) long.",
    },
    {
        "question": "Which F. Scott Fitzgerald novel features the character Jay Gatsby and is set during the Roaring Twenties?",
        "answer": "The Great Gatsby (1925).",
    },
    {
        "question": "What international sporting event originated in ancient Greece and was revived in Athens in 1896?",
        "answer": "The Olympic Games.",
    },
    {
        "question": "Which country won the first FIFA World Cup, and in what year was it held?",
        "answer": "Uruguay won the first FIFA World Cup, held in 1930.",
    },
    {
        "question": "Name the ancient artifact that was key to deciphering Egyptian hieroglyphics.",
        "answer": "The Rosetta Stone, discovered in 1799.",
    },
]

EVAL_QUESTIONS = [e["question"] for e in EVAL_TRIVIA]
print(f"{len(EVAL_TRIVIA)} evaluation questions reserved for testing.")

## 5. Load the Base Model

We load **Qwen 3.5 7B Instruct** with 4-bit QLoRA quantization. This fits comfortably on a T4 GPU (16 GB VRAM) and trains efficiently with LoRA rank 16.

> On a free-tier T4, loading takes ~2 minutes. On an A100, it's ~30 seconds.

In [ ]:
from konash.inference.local import LocalModelEngine

MODEL_NAME = "Qwen/Qwen3.5-7B-Instruct"

print(f"Loading {MODEL_NAME} with 4-bit QLoRA...")
t0 = time.time()

engine = LocalModelEngine(
    MODEL_NAME,
    device="auto",
    dtype="auto",
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    load_in_4bit=True,
    gradient_checkpointing=True,
)

load_time = time.time() - t0
print(f"\nModel loaded in {load_time:.1f}s")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory used: {allocated:.1f} GB")

### Quick Baseline Test

Before training, let's see how the base model answers trivia *without* search. This gives us a baseline to compare against after OAPL training.

In [ ]:
def baseline_answer(question: str) -> str:
    """Get a zero-shot answer from the base model (no search)."""
    messages = [
        {"role": "system", "content": "Answer the trivia question concisely and accurately."},
        {"role": "user", "content": question},
    ]
    response = engine.generate(messages, temperature=0.1, max_new_tokens=256)
    return response["content"]


print("Baseline (no search, no RL training):")
print("=" * 60)
baseline_results = []
for ev in EVAL_TRIVIA[:3]:  # Test first 3 for speed
    answer = baseline_answer(ev["question"])
    baseline_results.append({"question": ev["question"], "answer": answer})
    print(f"\nQ: {ev['question']}")
    print(f"Model: {answer[:200]}")
    print(f"Gold:  {ev['answer']}")

## 6. Stage 1: QA Synthesis

The model explores the Wikipedia corpus via vector search and synthesizes new multi-constraint trivia questions. These become the training data for OAPL.

This is KONASH's **agentic data synthesis** — the model writes its own curriculum.

In [ ]:
from konash.synthesis.pipeline import SynthesisPipeline
from konash.synthesis.qa import QuestionAnswerSynthesizer
from konash.synthesis.rollouts import RolloutGenerator
from konash.synthesis.dedup import DeduplicationAgent
from konash.synthesis.config import SynthesisTaskConfig, QualityFilterConfig


# Wrap engine.generate as an llm_fn callable
def llm_fn(messages, **kwargs):
    return engine.generate(messages, **kwargs)


# Configure synthesis for trivia
config = SynthesisTaskConfig(
    task_name="TriviaNight",
    seed_examples=len(SEED_TRIVIA),
    qa_max_steps=30,          # Trivia is usually quick to find
    qa_top_k=10,              # Moderate retrieval width
    qa_generation_count=16,   # Target 16 training questions
    solver_rollout_count=8,   # 8 rollouts per question
    solver_max_steps=20,      # Trivia needs fewer steps than domain QA
    solver_top_k=10,
    quality_filter=QualityFilterConfig(
        judge_model=None,     # Use heuristic judge (no external API needed)
    ),
)

# Build a string-returning search wrapper for the synthesizer
class StringSearchWrapper:
    def __init__(self, corpus):
        self._corpus = corpus
    def search(self, query, top_k=10, **kwargs):
        results = self._corpus.search(query, top_k=top_k)
        return [r["text"] if isinstance(r, dict) else str(r) for r in results]

synthesizer = QuestionAnswerSynthesizer(
    few_shot_examples=SEED_TRIVIA,
    vector_search_tool=StringSearchWrapper(corpus),
    generation_count=config.qa_generation_count,
    max_steps=config.qa_max_steps,
    llm_fn=llm_fn,
)

dedup_agent = DeduplicationAgent(evaluation_questions=EVAL_QUESTIONS)

pipeline = SynthesisPipeline(
    config=config,
    synthesizer=synthesizer,
    deduplication_agent=dedup_agent,
    evaluation_questions=EVAL_QUESTIONS,
)

In [ ]:
# Run Stage 1: Synthesize trivia questions from the corpus
print("Stage 1: Synthesizing trivia questions...")
print("=" * 60)

t0 = time.time()
doc_texts = [d["text"] for d in corpus.documents]
synthetic_examples = pipeline.run_stage_one(
    documents=doc_texts,
    num_examples=config.qa_generation_count,
)
stage1_time = time.time() - t0

print(f"\nGenerated {len(synthetic_examples)} trivia questions in {stage1_time:.1f}s")
print(f"(After deduplication against {len(EVAL_QUESTIONS)} eval questions)")
print()
for i, ex in enumerate(synthetic_examples, 1):
    print(f"  [{i}] Q: {ex.question}")
    print(f"      A: {ex.answer[:120]}{'...' if len(ex.answer) > 120 else ''}")
    print()

## 7. Stage 2: Rollout Generation & Filtering

For each synthetic question, we generate **8 independent solver rollouts**. Each rollout is a multi-step trajectory:

1. Search the corpus for relevant passages
2. Reason about the evidence
3. Optionally refine the search
4. Produce a final answer

Rollouts are scored against the reference answer (pass/fail). Questions where the model always passes or always fails are filtered out — we keep questions at the **learning frontier**.

In [ ]:
from konash.synthesis.rollouts import RolloutGenerator, RolloutGroup
from konash.synthesis.filters import PassRateFilter

# Build rollout generator with the local model
rollout_gen = RolloutGenerator(
    max_steps=config.solver_max_steps,
    top_k=config.solver_top_k,
    search_tool=corpus.vector_search,
    llm_fn=llm_fn,
)

# Generate rollouts for each synthetic question
print("Stage 2: Generating solver rollouts...")
print("=" * 60)

t0 = time.time()
rollout_groups = []
for i, ex in enumerate(synthetic_examples):
    print(f"  [{i+1}/{len(synthetic_examples)}] {ex.question[:65]}...")
    group = rollout_gen.generate_group(
        prompt=ex.question,
        reference_answer=ex.answer,
        num_rollouts=config.solver_rollout_count,
        seed=42 + i,
        qa_idx=i,
    )
    rollout_groups.append(group)
    passes = sum(1 for r in group.rollouts if r.passed)
    fails = sum(1 for r in group.rollouts if r.passed is False)
    print(f"    pass_rate={group.pass_rate:.0%} ({passes}P/{fails}F)")

stage2_time = time.time() - t0
total_rollouts = sum(g.size for g in rollout_groups)
print(f"\nGenerated {total_rollouts} rollouts in {stage2_time:.1f}s")

In [ ]:
# Pass-rate filtering: keep questions at the learning frontier
pass_filter = PassRateFilter(min_pass_rate=0.1, max_pass_rate=0.9)
filtered_groups = pass_filter.apply(rollout_groups)

print(f"Pass-rate filtering [0.1, 0.9]:")
print(f"  Before: {len(rollout_groups)} question groups")
print(f"  After:  {len(filtered_groups)} question groups")
print(f"  Dropped: {len(rollout_groups) - len(filtered_groups)} "
      f"(trivially easy or impossible)")
print()

for i, group in enumerate(filtered_groups, 1):
    print(f"  [{i}] pass_rate={group.pass_rate:.2f}  {group.prompt[:70]}...")

## 8. Build the RL Training Dataset

Convert the filtered rollout groups into the format expected by OAPL: prompts, rollout trajectories, and binary rewards.

In [ ]:
from konash.training.dataset import OfflineRolloutDataset

# Build training records from filtered groups
training_records = []
for group in filtered_groups:
    for rollout in group.rollouts:
        training_records.append({
            "prompt": group.prompt,
            "rollout": rollout.steps,
            "reward": 1.0 if rollout.passed else 0.0,
        })

dataset = OfflineRolloutDataset.from_rollouts(training_records)

positive = sum(1 for r in training_records if r["reward"] == 1.0)
negative = sum(1 for r in training_records if r["reward"] == 0.0)

print(f"Training dataset:")
print(f"  Prompt groups:  {len(dataset.prompts)}")
print(f"  Total rollouts: {len(training_records)}")
print(f"  Positive (1.0): {positive}")
print(f"  Negative (0.0): {negative}")
print(f"  Positive ratio: {positive / max(len(training_records), 1):.1%}")

## 9. Train with OAPL

Now we train the model using **Offline Advantage-weighted Policy Learning (OAPL)**. The key insight: this is fully offline — no interleaving with inference. The squared-advantage loss with KL regularization keeps the model close to its base policy while improving on the training signal.

**Critical detail:** Tool-output tokens (search results) are masked from the loss. The model only learns from tokens it actually generated.

$$\mathcal{L} = \left(\beta_{\text{KL}} \cdot \ln\frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)} - (r - \hat{V}^*)\right)^2$$

In [ ]:
from konash.training.oapl import OAPLTrainer

trainer = OAPLTrainer(
    beta_kl=0.1,      # KL regularization strength
    beta_value=0.05,   # Temperature for soft value estimation
)

# Training hyperparameters
LEARNING_RATE = 1e-5
NUM_ITERATIONS = 2

print("OAPL Training")
print("=" * 60)
print(f"  Iterations:     {NUM_ITERATIONS}")
print(f"  Learning rate:  {LEARNING_RATE}")
print(f"  beta_kl:        {trainer.beta_kl}")
print(f"  beta_value:     {trainer.beta_value}")
print(f"  LoRA rank:      16")
print(f"  Quantization:   4-bit QLoRA")
print()

In [ ]:
all_stats = []

for iteration in range(NUM_ITERATIONS):
    print(f"\n{'='*60}")
    print(f"  Iteration {iteration + 1}/{NUM_ITERATIONS}")
    print(f"{'='*60}")

    t0 = time.time()

    if iteration > 0:
        # Re-generate rollouts with the improved model
        print("  Re-generating rollouts with improved model...")
        rollout_groups_new = []
        for i, ex in enumerate(synthetic_examples):
            group = rollout_gen.generate_group(
                prompt=ex.question,
                reference_answer=ex.answer,
                num_rollouts=config.solver_rollout_count,
                seed=100 + iteration * 100 + i,
                qa_idx=i,
            )
            rollout_groups_new.append(group)

        filtered_groups = pass_filter.apply(rollout_groups_new)

        training_records = []
        for group in filtered_groups:
            for rollout in group.rollouts:
                training_records.append({
                    "prompt": group.prompt,
                    "rollout": rollout.steps,
                    "reward": 1.0 if rollout.passed else 0.0,
                })

        if not training_records:
            print("  No training data after filtering. Skipping.")
            continue

        dataset = OfflineRolloutDataset.from_rollouts(training_records)
        positive = sum(1 for r in training_records if r["reward"] == 1.0)
        print(f"  Dataset: {len(dataset.prompts)} groups, "
              f"{len(training_records)} rollouts, "
              f"{positive}/{len(training_records)} positive")

    # Run OAPL training epoch
    print("  Training...")
    epoch_stats = trainer.train_epoch_torch(
        dataset,
        engine,
        learning_rate=LEARNING_RATE,
    )

    elapsed = time.time() - t0
    all_stats.append({"iteration": iteration + 1, **epoch_stats})

    print(f"  Loss:           {epoch_stats['mean_loss']:.4f}")
    print(f"  Groups:         {epoch_stats['num_groups']}")
    print(f"  Rollouts:       {epoch_stats['num_rollouts']}")
    print(f"  Masked tokens:  {epoch_stats.get('masked_token_pct', 0):.1f}%")
    print(f"  Time:           {elapsed:.1f}s")

print("\n" + "=" * 60)
print("  Training Complete!")
print("=" * 60)

In [ ]:
# Save the trained LoRA adapter
CHECKPOINT_DIR = "./.konash/trivia-night/checkpoints/adapter"
engine.save_adapter(CHECKPOINT_DIR)

# Save training metadata
meta = {
    "base_model": MODEL_NAME,
    "task": "TriviaNight",
    "iterations": NUM_ITERATIONS,
    "corpus_articles": len(corpus_documents),
    "corpus_chunks": corpus.num_documents,
    "training_questions": len(dataset.prompts),
    "stats": all_stats,
}
meta_path = os.path.join(os.path.dirname(CHECKPOINT_DIR), "training_meta.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Checkpoint saved to {CHECKPOINT_DIR}")
print(f"Metadata saved to {meta_path}")

## 10. Training Summary

In [ ]:
print("Training Summary")
print("=" * 60)
print(f"  Model:            {MODEL_NAME}")
print(f"  Corpus:           {len(corpus_documents)} Wikipedia articles")
print(f"  Chunks indexed:   {corpus.num_documents}")
print(f"  Seed questions:   {len(SEED_TRIVIA)}")
print(f"  Synth questions:  {len(synthetic_examples)}")
print(f"  Training groups:  {len(dataset.prompts)}")
print()
print("  Per-iteration stats:")
for s in all_stats:
    print(f"    Iter {s['iteration']}: loss={s['mean_loss']:.4f}, "
          f"groups={s['num_groups']}, rollouts={s['num_rollouts']}")

## 11. Evaluate: Base Model vs. Trained Agent

Now let's compare the trained agent against the base model on our held-out evaluation set. The trained agent gets to **search the Wikipedia corpus** — just as it learned during training.

In [ ]:
from konash.agent import Agent as BaseAgent
from konash.harness.environment import Environment
from konash.plugins.compression import CompressionPlugin
from konash.plugins.control import StepBudgetPlugin


def solve_with_agent(question: str, max_steps: int = 15, top_k: int = 10) -> str:
    """Answer a question using the trained KONASH agent with search."""
    agent = BaseAgent(
        llm_client=engine,
        system_prompt=(
            "You are a trivia expert with access to a search tool. "
            "Search for evidence before answering. Be precise and concise."
        ),
        max_steps=max_steps,
    )

    def tool_executor(tool_call):
        query = ""
        if isinstance(tool_call, dict):
            query = tool_call.get("query", "") or tool_call.get("input", "")
            if not query:
                func = tool_call.get("function", {})
                args = func.get("arguments", {})
                if isinstance(args, str):
                    try:
                        args = json.loads(args)
                    except json.JSONDecodeError:
                        query = args
                if isinstance(args, dict):
                    query = args.get("query", "") or args.get("input", "")
        if not query:
            query = str(tool_call)

        results = corpus.search(query, top_k=top_k)
        text = "\n\n".join(
            f"[{j+1}] (score: {r.get('score', 0):.3f}) {r.get('text', '')[:400]}"
            for j, r in enumerate(results)
        )
        obs = {"role": "tool", "content": text}
        if isinstance(tool_call, dict) and tool_call.get("id"):
            obs["tool_call_id"] = tool_call["id"]
        return obs

    env = Environment(
        tool_executor=tool_executor,
        plugins=[
            CompressionPlugin(threshold_tokens=6000, target_tokens=3000),
            StepBudgetPlugin(max_steps=max_steps),
        ],
        available_tools=[{
            "type": "function",
            "function": {
                "name": "search",
                "description": "Search Wikipedia for relevant information.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "query": {"type": "string", "description": "Search query"}
                    },
                    "required": ["query"],
                },
            },
        }],
    )

    env.reset(prompt=question)
    result = env.run_episode(agent, max_steps=max_steps)
    return result.get("final_answer", "")

In [ ]:
print("Evaluation: Base Model vs. Trained Agent")
print("=" * 60)

results = []
for ev in EVAL_TRIVIA:
    q = ev["question"]
    gold = ev["answer"]

    # Base model (no search)
    base_answer = baseline_answer(q)

    # Trained agent (with search)
    agent_answer = solve_with_agent(q)

    results.append({
        "question": q,
        "gold": gold,
        "base": base_answer,
        "agent": agent_answer,
    })

    print(f"\nQ: {q}")
    print(f"Gold:  {gold}")
    print(f"Base:  {base_answer[:200]}")
    print(f"Agent: {agent_answer[:200]}")
    print("-" * 40)

### Score the Results

We use KONASH's heuristic evaluator to score both the base model and trained agent answers against the gold standard.

In [ ]:
from konash.synthesis.rollouts import _heuristic_evaluate

base_correct = 0
agent_correct = 0

print("\nScorecard")
print("=" * 60)
print(f"{'Question':<55} {'Base':>5} {'Agent':>5}")
print("-" * 67)

for r in results:
    base_pass = _heuristic_evaluate(r["base"], r["gold"])
    agent_pass = _heuristic_evaluate(r["agent"], r["gold"])
    base_correct += int(base_pass)
    agent_correct += int(agent_pass)

    q_short = r["question"][:53] + (".." if len(r["question"]) > 53 else "")
    print(f"{q_short:<55} {'PASS' if base_pass else 'FAIL':>5} "
          f"{'PASS' if agent_pass else 'FAIL':>5}")

n = len(results)
print("-" * 67)
print(f"{'TOTAL':<55} {base_correct}/{n:>3} {agent_correct}/{n:>3}")
print()
print(f"Base model accuracy:    {base_correct/n:.0%}")
print(f"Trained agent accuracy: {agent_correct/n:.0%}")
if agent_correct > base_correct:
    print(f"Improvement: +{agent_correct - base_correct} questions correct")

## 12. Interactive Demo

Try asking your own trivia questions! The trained agent will search the Wikipedia corpus and reason about the evidence before answering.

In [ ]:
# Try your own trivia questions!
YOUR_QUESTION = "What ancient wonder was located in the city of Babylon, and who supposedly built it?"

print(f"Q: {YOUR_QUESTION}")
print()
answer = solve_with_agent(YOUR_QUESTION)
print(f"Agent: {answer}")

In [ ]:
# Bonus: Use Parallel Thinking for higher accuracy
# This runs N independent rollouts and aggregates their answers
from konash.inference.parallel import ParallelThinkingEngine
from konash.inference.aggregation import GenerativeAggregator


def solve_parallel(question: str, num_rollouts: int = 5) -> str:
    """Solve with Parallel Thinking: N rollouts + generative aggregation."""
    answers = []
    for i in range(num_rollouts):
        ans = solve_with_agent(question)
        answers.append(ans)
        print(f"  Rollout {i+1}: {ans[:80]}...")

    # Simple majority vote for aggregation
    if not answers:
        return ""

    # Use the model to pick the best answer
    candidates = "\n".join(f"{i+1}. {a}" for i, a in enumerate(answers))
    messages = [
        {"role": "system", "content": "Pick the best answer from the candidates below. Return only the answer."},
        {"role": "user", "content": f"Question: {question}\n\nCandidate answers:\n{candidates}"},
    ]
    response = engine.generate(messages, temperature=0.1, max_new_tokens=256)
    return response["content"]


print("Parallel Thinking (5 rollouts + aggregation)")
print("=" * 60)
q = "What is the hottest desert on Earth, and on which continent is it located?"
print(f"Q: {q}\n")
answer = solve_parallel(q, num_rollouts=5)
print(f"\nFinal: {answer}")

## Using the High-Level API

Everything above can also be done with KONASH's one-liner API. Here's the equivalent using `konash.Agent`:

In [ ]:
# The entire pipeline above, as a one-liner:
#
#   import konash
#
#   agent = konash.Agent(
#       base_model="Qwen/Qwen3.5-7B-Instruct",
#       corpus="./trivia_corpus",
#       project="trivia-night",
#       load_in_4bit=True,
#       gradient_checkpointing=True,
#   )
#   agent.train(iterations=2, rollouts_per_example=8)
#   answer = agent.solve("What ancient wonder was in Babylon?", parallel_rollouts=10)
#   print(answer)
#
# This notebook shows the step-by-step internals for educational purposes.

print("Done! Your trivia agent is trained and ready.")
print(f"Checkpoint: {CHECKPOINT_DIR}")
print()
print("Next steps:")
print("  - Add more Wikipedia articles to the corpus for broader coverage")
print("  - Run a 3rd training iteration for further improvement")
print("  - Use Parallel Thinking (N=10-20) for near-deterministic accuracy")
print("  - Deploy with vLLM + LoRA hot-swapping for production serving")